In [1]:
!pip install yfinance

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 71.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 56.6 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 12.0
    Uninstalling websockets-12.0:
      Successfully uninstalled websockets-12.0
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.


In [2]:
# ============================================================
# NB01_Universe_Ingestion
# Purpose:
#   1. Read Nifty 500 constituents
#   2. Classify financial companies
#   3. Apply manual overrides
#   4. Add Nifty 50 market index
#   5. Create bronze_universe_snapshot
# ============================================================

import pandas as pd
import numpy as np
import re
from datetime import datetime

RAW_UNIVERSE_PATH = "Files/reference/nifty500_raw.csv"
OVERRIDES_PATH = "Files/reference/financial_overrides.csv"

RUN_DATE = datetime.utcnow().strftime("%Y-%m-%d")
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 4, Finished, Available, Finished, False)

In [3]:

# ------------------------------------------------------------
# 1. Read the Nifty 500 raw file
# ------------------------------------------------------------

raw_spark = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(RAW_UNIVERSE_PATH)
)

raw_pd = raw_spark.toPandas()

print("Original columns:")
print(raw_pd.columns.tolist())




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 5, Finished, Available, Finished, False)

Original columns:
['Company Name', 'Industry', 'Symbol', 'Series', 'ISIN Code']


In [4]:
# ------------------------------------------------------------
# 2. Normalize column names
# ------------------------------------------------------------

def normalize_column_name(column_name):
    column_name = column_name.strip().lower()
    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    return column_name.strip("_")


raw_pd.columns = [
    normalize_column_name(c)
    for c in raw_pd.columns
]

print("Normalized columns:")
print(raw_pd.columns.tolist())




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 6, Finished, Available, Finished, False)

Normalized columns:
['company_name', 'industry', 'symbol', 'series', 'isin_code']


In [5]:
# ------------------------------------------------------------
# 3. Check required columns
# ------------------------------------------------------------

if "symbol" not in raw_pd.columns:
    raise ValueError(
        "The input file must contain a Symbol column. "
        f"Available columns: {raw_pd.columns.tolist()}"
    )

if "industry" not in raw_pd.columns:
    print("Industry column not found. Creating an empty column.")
    raw_pd["industry"] = ""

if "company_name" not in raw_pd.columns:
    print("Company Name column not found. Using symbol as company name.")
    raw_pd["company_name"] = raw_pd["symbol"]




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 7, Finished, Available, Finished, False)

In [6]:
# ------------------------------------------------------------
# 4. Keep equity securities only
# ------------------------------------------------------------

if "series" in raw_pd.columns:
    raw_pd["series"] = (
        raw_pd["series"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Nifty constituent files normally use EQ.
    # If Series is unavailable, this filter is skipped.
    raw_pd = raw_pd[
        raw_pd["series"].isin(["EQ", ""])
    ].copy()




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 8, Finished, Available, Finished, False)

In [7]:
# ------------------------------------------------------------
# 5. Clean symbols and names
# ------------------------------------------------------------

stocks = raw_pd[[
    "symbol",
    "company_name",
    "industry"
]].copy()

stocks["symbol"] = (
    stocks["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

stocks["company_name"] = (
    stocks["company_name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

stocks["industry"] = (
    stocks["industry"]
    .fillna("")
    .astype(str)
    .str.strip()
)

stocks = stocks[
    (stocks["symbol"] != "") &
    (stocks["symbol"] != "NAN")
].copy()

stocks = stocks.drop_duplicates(subset=["symbol"])




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 9, Finished, Available, Finished, False)

In [8]:
# ------------------------------------------------------------
# 6. Apply automatic financial keyword classification
# ------------------------------------------------------------

financial_pattern = (
    r"\b(bank|banking|finance|financial|insurance|nbfc|"
    r"microfinance|mortgage)\b"
    r"|asset\s*management"
    r"|housing\s*finance"
    r"|capital\s*markets"
)

industry_text = (
    stocks["industry"]
    .str.lower()
    .str.replace("&", "and", regex=False)
    .str.replace("-", " ", regex=False)
)

industry_unknown = (
    industry_text.isna() |
    industry_text.str.strip().eq("")
)

auto_is_financial = industry_text.str.contains(
    financial_pattern,
    regex=True,
    na=False
)

stocks["auto_is_financial"] = auto_is_financial
stocks["industry_unknown"] = industry_unknown

stocks["classification_status"] = np.select(
    [
        stocks["industry_unknown"],
        stocks["auto_is_financial"]
    ],
    [
        "NEEDS_REVIEW",
        "KEYWORD_FINANCIAL"
    ],
    default="KEYWORD_NON_FINANCIAL"
)

stocks["is_financial"] = stocks["auto_is_financial"].astype(bool)




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 10, Finished, Available, Finished, False)

/tmp/ipykernel_8194/2012384315.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  auto_is_financial = industry_text.str.contains(


In [9]:
# ------------------------------------------------------------
# 7. Read manual overrides
# ------------------------------------------------------------

try:
    overrides_spark = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(OVERRIDES_PATH)
    )

    overrides = overrides_spark.toPandas()

    overrides.columns = [
        normalize_column_name(c)
        for c in overrides.columns
    ]

except Exception as error:
    print("No override file could be read.")
    print("Reason:", str(error))

    overrides = pd.DataFrame(
        columns=[
            "symbol",
            "override_action",
            "override_reason"
        ]
    )




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 11, Finished, Available, Finished, False)

In [10]:
# ------------------------------------------------------------
# 8. Clean and validate manual overrides
# ------------------------------------------------------------

if len(overrides) > 0:

    required_override_columns = [
        "symbol",
        "override_action",
        "override_reason"
    ]

    missing_override_columns = [
        c for c in required_override_columns
        if c not in overrides.columns
    ]

    if missing_override_columns:
        raise ValueError(
            "Missing columns in financial_overrides.csv: "
            f"{missing_override_columns}"
        )

    overrides["symbol"] = (
        overrides["symbol"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    overrides["override_action"] = (
        overrides["override_action"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    overrides["override_reason"] = (
        overrides["override_reason"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    allowed_actions = {
        "EXCLUDE_FINANCIAL",
        "INCLUDE_NON_FINANCIAL",
        "EXCLUDE_STOCK"
    }

    invalid_actions = set(
        overrides["override_action"]
    ) - allowed_actions

    if invalid_actions:
        raise ValueError(
            f"Invalid override actions: {invalid_actions}. "
            f"Allowed actions: {allowed_actions}"
        )

    overrides = overrides.drop_duplicates(
        subset=["symbol"],
        keep="last"
    )

else:
    overrides = pd.DataFrame(
        columns=[
            "symbol",
            "override_action",
            "override_reason"
        ]
    )




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 12, Finished, Available, Finished, False)

In [11]:
# ------------------------------------------------------------
# 9. Join manual overrides to the stock universe
# ------------------------------------------------------------

stocks = stocks.merge(
    overrides,
    on="symbol",
    how="left"
)

stocks["override_action"] = (
    stocks["override_action"]
    .fillna("")
    .astype(str)
    .str.upper()
)

stocks["override_reason"] = (
    stocks["override_reason"]
    .fillna("")
    .astype(str)
)




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 13, Finished, Available, Finished, False)

In [12]:
# ------------------------------------------------------------
# 10. Apply manual overrides
# ------------------------------------------------------------

exclude_financial_mask = (
    stocks["override_action"] == "EXCLUDE_FINANCIAL"
)

include_non_financial_mask = (
    stocks["override_action"] == "INCLUDE_NON_FINANCIAL"
)

exclude_stock_mask = (
    stocks["override_action"] == "EXCLUDE_STOCK"
)

# Financial override
stocks.loc[
    exclude_financial_mask,
    "is_financial"
] = True

# Non-financial override
stocks.loc[
    include_non_financial_mask,
    "is_financial"
] = False

# Classification status
stocks.loc[
    stocks["override_action"] != "",
    "classification_status"
] = "MANUAL_OVERRIDE"




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 14, Finished, Available, Finished, False)

In [13]:
# ------------------------------------------------------------
# 11. Create Yahoo Finance ticker symbols
# ------------------------------------------------------------

# These are equity symbols only.
# Nifty 50 is added separately with ^NSEI later.

stocks["yahoo_symbol"] = (
    stocks["symbol"].astype(str) + ".NS"
)

stocks["instrument_type"] = "EQUITY"
stocks["is_market_index"] = False




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 15, Finished, Available, Finished, False)

In [14]:
# ------------------------------------------------------------
# 12. Create the stock-screen eligibility flag
# ------------------------------------------------------------

# Conservative treatment:
# - Financial companies are excluded.
# - Unknown industries require manual review.
# - EXCLUDE_STOCK overrides are excluded.

stocks["eligible_for_stock_screen"] = (
    (stocks["instrument_type"] == "EQUITY") &
    (~stocks["is_financial"]) &
    (~stocks["industry_unknown"]) &
    (~exclude_stock_mask)
)




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 16, Finished, Available, Finished, False)

In [15]:
# ------------------------------------------------------------
# 13. Add Nifty 50 separately
# ------------------------------------------------------------

nifty50_row = pd.DataFrame([{
    "symbol": "NIFTY50",
    "yahoo_symbol": "^NSEI",
    "company_name": "Nifty 50",
    "industry": "Index",
    "instrument_type": "INDEX",
    "is_market_index": True,
    "is_financial": False,
    "auto_is_financial": False,
    "industry_unknown": False,
    "classification_status": "MARKET_INDEX",
    "override_action": "",
    "override_reason": "",
    "eligible_for_stock_screen": False
}])




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 17, Finished, Available, Finished, False)

In [16]:
# ------------------------------------------------------------
# 14. Select and combine final columns
# ------------------------------------------------------------

final_columns = [
    "symbol",
    "yahoo_symbol",
    "company_name",
    "industry",
    "instrument_type",
    "is_market_index",
    "is_financial",
    "auto_is_financial",
    "industry_unknown",
    "classification_status",
    "override_action",
    "override_reason",
    "eligible_for_stock_screen"
]

stocks_final = stocks[final_columns].copy()
nifty50_final = nifty50_row[final_columns].copy()

universe_final = pd.concat(
    [stocks_final, nifty50_final],
    ignore_index=True
)




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 18, Finished, Available, Finished, False)

In [17]:
# ------------------------------------------------------------
# 15. Add audit columns
# ------------------------------------------------------------

universe_final["universe_date"] = RUN_DATE
universe_final["source_file"] = "nifty500_raw.csv"
universe_final["ingestion_timestamp"] = RUN_TIMESTAMP

# Replace pandas missing values with Python None
universe_final = universe_final.where(
    pd.notnull(universe_final),
    None
)




StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 19, Finished, Available, Finished, False)

In [18]:
# ------------------------------------------------------------
# 16. Save as a Delta table in the Lakehouse
# ------------------------------------------------------------

universe_spark = spark.createDataFrame(universe_final)

(
    universe_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_universe_snapshot")
)

print("Universe ingestion completed.")
print(f"Total rows: {len(universe_final)}")

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 20, Finished, Available, Finished, False)

Universe ingestion completed.
Total rows: 500


In [19]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_universe_snapshot
        WHERE symbol = 'NIFTY50'
    """)
)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d987555e-c36a-4615-ab07-0691a02ee343)

In [20]:
display(
    spark.sql("""
        SELECT
            instrument_type,
            is_financial,
            eligible_for_stock_screen,
            COUNT(*) AS stock_count
        FROM bronze_universe_snapshot
        GROUP BY
            instrument_type,
            is_financial,
            eligible_for_stock_screen
        ORDER BY
            instrument_type,
            is_financial,
            eligible_for_stock_screen
    """)
)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7d11367-2c48-4f37-bf0c-e5503a71133b)

In [21]:
display(
    spark.sql("""
        SELECT
            symbol,
            company_name,
            industry,
            classification_status,
            override_action,
            is_financial,
            eligible_for_stock_screen
        FROM bronze_universe_snapshot
        WHERE instrument_type = 'EQUITY'
          AND is_financial = true
        ORDER BY symbol
    """)
)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a3d3699-d778-41b3-ae92-37a0c718ed65)

In [22]:
display(
    spark.sql("""
        SELECT
            symbol,
            company_name,
            industry,
            classification_status,
            eligible_for_stock_screen
        FROM bronze_universe_snapshot
        WHERE classification_status = 'NEEDS_REVIEW'
    """)
)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6c34dc0-dca4-41e3-9b8a-045299828cb7)

In [23]:
stock_universe = spark.sql("""
    SELECT
        symbol,
        yahoo_symbol,
        company_name,
        industry
    FROM bronze_universe_snapshot
    WHERE instrument_type = 'EQUITY'
      AND eligible_for_stock_screen = true
""")

display(stock_universe.limit(10))

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5909db68-a58c-43fe-9d14-3561e66fdd90)

In [24]:
market_instrument = spark.sql("""
    SELECT
        symbol,
        yahoo_symbol,
        company_name
    FROM bronze_universe_snapshot
    WHERE instrument_type = 'INDEX'
      AND symbol = 'NIFTY50'
""")

display(market_instrument)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ef4b100-c2ca-4339-8af9-532bfd5c0da6)

In [25]:
!pip install yfinance

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 27, Finished, Available, Finished, False)

In [26]:
import yfinance as yf

nifty = yf.download(
    "^NSEI",
    period="5y",
    auto_adjust=True,
    progress=False
)

StatementMeta(, 98bfd439-9094-43f0-80ba-49ad72724375, 28, Finished, Available, Finished, False)